In [ ]:
# Purpose - plots fields and cuts for a single file

findex = 250 #350 #280 #260 #-7 #140 #230 #190 #230 #140 #-7 

step_choice = "all" #"penultimate" #"last" #"last" #all

In [ ]:
# Imports and Folder Selection

%load_ext autoreload
%autoreload 2

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys
sys.path.append('../../v2_proc_files')
from useful_funcs import *
from TristanRun import TristanRun
from FieldLineHelper import FieldLineHelper 

plt.close('all')

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq

cc = run.cc
B0 = run.B0

tstep_choices = {"first": [run.steps[0]], "last": [run.steps[-1]], "all": run.steps[:-1], "penultimate": [run.steps[-2]],
                 "last_two": run.steps[-2:]}

my_steps = tstep_choices[step_choice]
#my_steps = list(range(85, 681, 5))
#my_steps = [85]
print(len(my_steps))
#my_steps = run.steps[-6:]

#boilerplate = run.boilerplate + f" tstep = {my_steps[0]}"

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields", includes_pressure = False)

In [ ]:
ts = 0
rect = [10, run.xlim - 10, 0, run.ylim]

cs = fq_cut(d.fields, ts, rect)
plot_fq3(cs, cs.ndens, vabsmax = 5, cmap = 'viridis', positive = True, downsample = 4)

In [ ]:
import os

filename = f'{run.path}/crossings_data.npy'

# Check if the file already exists
if os.path.exists(filename):
    print(f"Loading existing data from {filename}...")
    crossings = np.load(filename).tolist()
else:
    print("File not found. Generating crossings data...")
    crossings = []

    for ts in range(len(my_steps)):
        YVAL = run.ylim / 2
        midcut = d.fields.isel(t=ts).sel(y=run.ylim / 2, method='nearest')

        by_data = midcut.by.squeeze().values
        zero_crossings = np.where(np.diff(np.sign(by_data)))[0]
        
        # Interpolate to find more precise zero crossing
        if len(zero_crossings) > 0:
            idx = zero_crossings[0]
            # Linear interpolation: x = x1 - y1 * (x2 - x1) / (y2 - y1)
            x1, x2 = midcut.x.values[idx], midcut.x.values[idx + 1]
            y1, y2 = by_data[idx], by_data[idx + 1]
            zero_cross_x_interp = x1 - y1 * (x2 - x1) / (y2 - y1)
        else:
            zero_cross_x_interp = np.nan

        # Logging logic
        if len(zero_crossings) == 0:
            print(f"ts {ts}: No zero crossings found")
        elif len(zero_crossings) > 1:
            # We already calculated interp for the first one above
            zero_cross_x_values = midcut.x.values[zero_crossings]
            print(f"ts {ts}: Multiple zero crossings found at x = {zero_cross_x_values}")
        else:
            print(f"ts {ts}: Zero crossing at x = {zero_cross_x_interp}")

        crossings.append(zero_cross_x_interp)

    # Save the result for next time
    np.save(filename, np.array(crossings))
    print(f"Data saved to {filename}.")

plt.close('all')

In [ ]:
plt.close('all')

plt.title(f"folder = {findex}, mime = {run.mime}, ylim = {run.ylim}")
plt.plot(my_steps, crossings, marker = 'o')
plt.axhline(run.xlim / 2, color='red', linestyle='--', label='xlim / 2')
plt.legend()
plt.xlabel('Time Step')
plt.ylabel('Current Sheet $x$ position')

plt.show()

In [ ]:
findices= [190, 230, -7, "crossings_data_pad50", "crossings_data_pad100", "crossings_data_pad50_Ez0", "crossings_data_doublex", 270, 280, 290, 350, 450]
descrs = ["Ez 0 BC", "Ez 0.1 B0 BC", "nersc run, 0.1 B0 BC", "normal padding, 0.1 vA B0 BC", 
          "doubled padding, 0.1 vA B0 BC", "normal padding, 0 B0 BC", "doubled x-extent, 0 B0 BC", "vay pusher 0 B0 BC", "dynamic Ez B0 BC", 'absorb BC 0', 'processor 2 x 480', 'low beta']

findices = [-7, 450]
descrs = ['nersc run, 0.1 B0 BC', 'low beta']


if (len(findices) != len(descrs)):
    raise ValueError("Length of findices and descrs must be the same")

for i in range(len(findices)):
    findex = findices[i]
    if isinstance(findex, str):
        filename = findex
        findex = 270
        cs_file = f"{TristanRun(findex).path}/{filename}.npy"
    else:
        run_i = TristanRun(findex)
        cs_file = f"{run_i.path}/crossings_data.npy"
        print(run_i.output_interval)

    if findex == findices[0]:
        plt.close("all")
        plt.figure()

    if os.path.exists(cs_file):
        cs_shift = np.load(cs_file)
        steps_i = range(len(cs_shift))
        n = min(len(steps_i), len(cs_shift))
        plt.plot(steps_i[:n], cs_shift[:n] - run_i.xlim / 2, marker="o", ms=2, lw=1, label=f"run {findex}, ylim = {run_i.ylim}, {descrs[i]}")
    else:
        raise FileNotFoundError(f"File {cs_file} not found for run {findex}")
        print(f"Missing file: {cs_file}")


plt.xlabel("Time Step")
plt.ylabel("Change in CS x Position (comp)")
plt.title("CS shift comparison")
#plt.legend()
plt.grid(alpha=0.3)
plt.legend(bbox_to_anchor=(1.04, 0.5), loc="center left")
plt.tight_layout()
plt.show()
#plt.legend()

In [ ]:
findices= [-7, 250]
descrs = ["nersc run", 'processor 2 x 480']


if (len(findices) != len(descrs)):
    raise ValueError("Length of findices and descrs must be the same")

for i in range(len(findices)):
    findex = findices[i]
    if isinstance(findex, str):
        filename = findex
        findex = 270
        cs_file = f"{TristanRun(findex).path}/{filename}.npy"
    else:
        run_i = TristanRun(findex)
        cs_file = f"{run_i.path}/crossings_data.npy"
        print(run_i.output_interval)

    if findex == findices[0]:
        plt.close("all")
        plt.figure()

    if os.path.exists(cs_file):
        cs_shift = np.load(cs_file)
        steps_i = range(len(cs_shift))
        n = min(len(steps_i), len(cs_shift))
        plt.plot(steps_i[:n], cs_shift[:n] - run_i.xlim / 2, marker="o", ms=2, lw=1, label=f"run {findex}, ylim = {run_i.ylim}, {descrs[i]}")
    else:
        raise FileNotFoundError(f"File {cs_file} not found for run {findex}")
        print(f"Missing file: {cs_file}")


plt.xlabel("Time Step")
plt.ylabel("Change in CS x Position (comp)")
plt.title("CS shift comparison")
#plt.legend()
plt.grid(alpha=0.3)
#plt.legend(bbox_to_anchor=(1.04, 0.5), loc="center left")
plt.legend()
plt.tight_layout()
plt.show()
#plt.legend()